In [1]:
# ============================================================
# Ablation: 2 missing cells for the paper's 2x2 table
#
# What we already have (no retraining needed):
#   [Naive  pruning, no KD]  → 78.29%  (cifar100-dense-sparse-vit.ipynb)
#   [Naive  pruning, + KD]   → 77.28%  (cifar-100-sparsevit-kd.ipynb)
#   [Prog.  pruning, + KD]   → 89.60%  (sparse-kd-cifar-100.ipynb)
#
# What we need:
#   [Prog.  pruning, no KD]  → ???     ← Run 1  (~3.5h)
#   [Naive  pruning, + FeatKD] → ???   ← Run 2  (~3.5h)
#
# Run 1 answers: does progressive pruning alone explain the gain?
# Run 2 answers: does feature KD alone explain the gain?
# Together they prove the combination is what matters.
#
# Total runtime: ~7h on P100 (within single session limit)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy, math, json
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ============================================================
# DATA
# ============================================================

def get_cifar100(batch_size=64):
    train_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])
    train_ds = torchvision.datasets.CIFAR100('./data', train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR100('./data', train=False, download=True, transform=test_tf)
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    return train_dl, test_dl

train_dl, test_dl = get_cifar100()

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        out    = model(x)
        logits = out[0] if isinstance(out, tuple) else out
        correct += (logits.argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total

# ============================================================
# ARCHITECTURE 1: ProgressiveSparseViT
# Used for Run 1 (progressive pruning, CE only — no KD)
# Same architecture as the 89.60% model, KD just disabled
# ============================================================

class ProgressiveSparseViT(nn.Module):
    def __init__(self, base_model, keep_ratio_1=0.85, keep_ratio_2=0.50):
        super().__init__()
        self.base = base_model
        self.kr1  = keep_ratio_1
        self.kr2  = keep_ratio_2
        layers = list(self.base.encoder.layers.children())
        self.stage1 = nn.Sequential(*layers[0:4])
        self.stage2 = nn.Sequential(*layers[4:8])
        self.stage3 = nn.Sequential(*layers[8:12])

    def _prune(self, x, keep_ratio, num_patches):
        B, _, C = x.shape
        k       = max(1, int(num_patches * keep_ratio))
        scores  = x[:, 1:].norm(dim=-1)
        topk    = torch.topk(scores, k, dim=1).indices + 1
        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        idx     = torch.cat([cls_idx, topk], dim=1)
        return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, C))

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape
        cls = self.base.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.base.encoder.pos_embedding
        x   = self.base.encoder.dropout(x)
        x   = self.stage1(x)
        x   = self._prune(x, self.kr1, N)
        feat_mid = x[:, 0]
        k1  = max(1, int(N * self.kr1))
        x   = self.stage2(x)
        x   = self._prune(x, self.kr2 / self.kr1, k1)
        feat_late = x[:, 0]
        x   = self.stage3(x)
        x   = self.base.encoder.ln(x)
        cls_out = x[:, 0]
        return self.base.heads(cls_out), feat_mid, feat_late, cls_out

# ============================================================
# ARCHITECTURE 2: NaiveSparseViT with feature outputs
# Used for Run 2 (one-shot 50% pruning + feature KD)
# Same naive pruning as original, but returns features for KD
# ============================================================

class NaiveSparseViTWithFeatures(nn.Module):
    """One-shot token pruning at input (original approach),
    but now returns intermediate CLS features for feature KD.
    Isolates whether feature KD alone explains the 89.60% result
    without progressive pruning."""
    def __init__(self, base_model, keep_ratio=0.50):
        super().__init__()
        self.base = base_model
        self.kr   = keep_ratio
        layers = list(self.base.encoder.layers.children())
        self.stage1 = nn.Sequential(*layers[0:4])
        self.stage2 = nn.Sequential(*layers[4:8])
        self.stage3 = nn.Sequential(*layers[8:12])

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape
        cls = self.base.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.base.encoder.pos_embedding
        x   = self.base.encoder.dropout(x)

        # ONE-SHOT prune before any transformer block
        k       = max(1, int(N * self.kr))
        scores  = x[:, 1:].norm(dim=-1)
        topk    = torch.topk(scores, k, dim=1).indices + 1
        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        idx     = torch.cat([cls_idx, topk], dim=1)
        x       = torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, C))

        # Run all 3 stages on the pruned sequence
        x = self.stage1(x);  feat_mid  = x[:, 0]
        x = self.stage2(x);  feat_late = x[:, 0]
        x = self.stage3(x)
        x = self.base.encoder.ln(x)
        cls_out = x[:, 0]
        return self.base.heads(cls_out), feat_mid, feat_late, cls_out

# ============================================================
# TEACHER WRAPPER
# ============================================================

class TeacherWithFeatures(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        layers = list(self.base.encoder.layers.children())
        self.stage1 = nn.Sequential(*layers[0:4])
        self.stage2 = nn.Sequential(*layers[4:8])
        self.stage3 = nn.Sequential(*layers[8:12])

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape
        cls = self.base.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.base.encoder.pos_embedding
        x   = self.base.encoder.dropout(x)
        x   = self.stage1(x);  feat_mid  = x[:, 0]
        x   = self.stage2(x);  feat_late = x[:, 0]
        x   = self.stage3(x)
        x   = self.base.encoder.ln(x)
        cls_out = x[:, 0]
        return self.base.heads(cls_out), feat_mid, feat_late, cls_out

# ============================================================
# SHARED TRAINING FUNCTION
# use_kd=False  → CE + label smoothing only  (Run 1)
# use_kd=True   → CE + logit KD + feature KD (Run 2)
# ============================================================

def train_run(student, teacher, run_name, epochs=15, use_kd=True,
              T=4.0, feat_weight=0.3, base_lr=5e-5, warmup_epochs=3):

    print(f'\n{"-"*55}')
    print(f'  {run_name}  |  KD={use_kd}  |  {epochs} epochs')
    print(f'{"-"*55}')

    student.to(device)
    ce_loss   = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(student.parameters(), lr=base_lr, weight_decay=0.05)

    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        min_frac = 1e-6 / base_lr
        return min_frac + 0.5*(1 - min_frac)*(1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.amp.GradScaler('cuda')
    best_acc  = 0.0

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0
        alpha = 0.70 - 0.20 * (epoch / max(1, epochs - 1))  # 0.70→0.50

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)

            # MixUp (only when KD active, so teacher & student see same input)
            if use_kd:
                lam   = np.random.beta(0.4, 0.4)
                idx   = torch.randperm(x.size(0), device=device)
                x_in  = lam * x + (1 - lam) * x[idx]
                y_a, y_b = y, y[idx]
            else:
                x_in, y_a, y_b, lam = x, y, y, 1.0

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                s_logits, s_mid, s_late, _ = student(x_in)
                ce = lam*ce_loss(s_logits, y_a) + (1-lam)*ce_loss(s_logits, y_b)

                if use_kd:
                    with torch.no_grad():
                        t_logits, t_mid, t_late, _ = teacher(x_in)
                    kd_logit = F.kl_div(
                        F.log_softmax(s_logits/T, dim=1),
                        F.softmax(t_logits/T,    dim=1),
                        reduction='batchmean'
                    ) * (T*T)
                    kd_feat = 0.5*(
                        F.mse_loss(F.normalize(s_mid,  dim=-1), F.normalize(t_mid,  dim=-1)) +
                        F.mse_loss(F.normalize(s_late, dim=-1), F.normalize(t_late, dim=-1))
                    )
                    loss = (1-alpha)*ce + alpha*kd_logit + feat_weight*kd_feat
                else:
                    loss = ce

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        scheduler.step()
        acc = evaluate(student)
        print(f'  Epoch {epoch+1:02d}/{epochs} | '
              f'Loss: {total_loss/len(train_dl):.4f} | ValAcc: {acc:.4f}')

        if acc > best_acc:
            best_acc = acc
            torch.save(student.state_dict(), f'{run_name}_best.pt')
            print(f'    ✓ Best: {best_acc:.4f}')

    return best_acc

# ============================================================
# LOAD TEACHER
# ============================================================

base_teacher = vit_b_16(weights=None)
base_teacher.heads.head = nn.Linear(base_teacher.heads.head.in_features, 100)
base_teacher.load_state_dict(
    torch.load('/kaggle/input/datasets/nagatouzumakiii/densevit-cifar100/dense_cifar100_best.pt',
               map_location=device)
)
base_teacher.to(device).eval()
teacher_acc = evaluate(base_teacher)
print(f'Teacher accuracy: {teacher_acc:.4f}')
teacher = TeacherWithFeatures(base_teacher).to(device).eval()

# ============================================================
# RUN 1: Progressive pruning, CE only (no KD)
# Question: does progressive pruning alone explain the gain?
# ============================================================

student1 = ProgressiveSparseViT(copy.deepcopy(base_teacher), keep_ratio_1=0.85, keep_ratio_2=0.50)
acc_prog_nokd = train_run(student1, teacher,
                          run_name='progressive_nokd',
                          epochs=15, use_kd=False)
json.dump({'model': 'Progressive Sparse, no KD', 'accuracy': acc_prog_nokd},
          open('progressive_nokd_result.json', 'w'))
print(f'\nRun 1 done → {acc_prog_nokd:.4f}')

# Free VRAM before Run 2
del student1
torch.cuda.empty_cache()

# ============================================================
# RUN 2: Naive (one-shot) pruning + full feature KD
# Question: does feature KD alone explain the gain,
#           without progressive pruning?
# ============================================================

student2 = NaiveSparseViTWithFeatures(copy.deepcopy(base_teacher), keep_ratio=0.50)
acc_naive_kd = train_run(student2, teacher,
                         run_name='naive_sparse_featkd',
                         epochs=15, use_kd=True)
json.dump({'model': 'Naive Sparse + Feature KD', 'accuracy': acc_naive_kd},
          open('naive_sparse_featkd_result.json', 'w'))
print(f'\nRun 2 done → {acc_naive_kd:.4f}')

# ============================================================
# FINAL 2x2 ABLATION TABLE
# ============================================================

print('\n' + '='*58)
print('  ABLATION TABLE (2x2)')
print('='*58)
print(f'  {"Model":<35} {"Tokens":>7}  {"Acc":>7}')
print('-'*58)
print(f'  {"Dense ViT-B/16 (Teacher)":<35} {"100%":>7}  {teacher_acc:.4f}')
print(f'  {"Naive Sparse, CE only":<35} {"50%":>7}  0.7829 (existing)')
print(f'  {"Naive Sparse + Logit KD":<35} {"50%":>7}  0.7728 (existing)')
print(f'  {"Naive Sparse + Feature KD (Run 2)":<35} {"50%":>7}  {acc_naive_kd:.4f}')
print(f'  {"Progressive Sparse, CE only (Run 1)":<35} {"50%":>7}  {acc_prog_nokd:.4f}')
print(f'  {"Progressive Sparse + Feature KD":<35} {"50%":>7}  0.8960 (existing)')
print('='*58)
print()
print('Key gaps for paper:')
print(f'  KD contribution:          0.8960 - {acc_prog_nokd:.4f} = {0.8960 - acc_prog_nokd:.4f}')
print(f'  Progressive contribution: 0.8960 - {acc_naive_kd:.4f} = {0.8960 - acc_naive_kd:.4f}')
print(f'  vs naive baseline:        0.8960 - 0.7829          = {0.8960 - 0.7829:.4f}')

Device: cuda


100%|██████████| 169M/169M [00:05<00:00, 30.4MB/s] 


Teacher accuracy: 0.8703

-------------------------------------------------------
  progressive_nokd  |  KD=False  |  15 epochs
-------------------------------------------------------
  Epoch 01/15 | Loss: 0.9555 | ValAcc: 0.8731
    ✓ Best: 0.8731
  Epoch 02/15 | Loss: 0.9151 | ValAcc: 0.8748
    ✓ Best: 0.8748
  Epoch 03/15 | Loss: 0.9353 | ValAcc: 0.8555
  Epoch 04/15 | Loss: 0.9326 | ValAcc: 0.8562
  Epoch 05/15 | Loss: 0.9117 | ValAcc: 0.8656
  Epoch 06/15 | Loss: 0.8959 | ValAcc: 0.8605
  Epoch 07/15 | Loss: 0.8774 | ValAcc: 0.8649
  Epoch 08/15 | Loss: 0.8598 | ValAcc: 0.8696
  Epoch 09/15 | Loss: 0.8403 | ValAcc: 0.8714
  Epoch 10/15 | Loss: 0.8276 | ValAcc: 0.8703
  Epoch 11/15 | Loss: 0.8127 | ValAcc: 0.8857
    ✓ Best: 0.8857
  Epoch 12/15 | Loss: 0.8037 | ValAcc: 0.8845
  Epoch 13/15 | Loss: 0.7972 | ValAcc: 0.8875
    ✓ Best: 0.8875
  Epoch 14/15 | Loss: 0.7933 | ValAcc: 0.8886
    ✓ Best: 0.8886
  Epoch 15/15 | Loss: 0.7906 | ValAcc: 0.8898
    ✓ Best: 0.8898

Run 1 done 